In [1]:
import subprocess, time, os

subprocess.run('apt-get install -y zstd lshw', shell=True, capture_output=True)
result = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-300:])
print(result.stderr[-200:])



r to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [2]:
import os, subprocess, time, urllib.request

os.environ["OLLAMA_MODELS"] = "/kaggle/temp/ollama"   # ~9 GB model lands on scratch
os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)

subprocess.Popen(["ollama", "serve"],
                 stdout=open("/tmp/ollama.log", "w"),
                 stderr=subprocess.STDOUT)

# wait for the API to answer
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2); print("Ollama up"); break
    except Exception:
        time.sleep(2)
else:
    print("Ollama did NOT start — check /tmp/ollama.log")


Ollama up


In [ ]:
import subprocess, sys

process = subprocess.Popen(
    ['ollama', 'pull', 'qwen2.5:32b'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f"\nExit code: {process.returncode}")

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling eabc98a9bcbf:   0% ▕                  ▏ 922 KB/ 19 GB                  pulling manifest 
pulling eabc98a9bcbf:   0% ▕                  ▏  39 MB/ 19 GB                  pulling manifest 
pulling eabc98a9bcbf:   1% ▕                  ▏ 108 MB/ 19 GB                  pulling manifest 
pulling eabc98a9bcbf:   1% ▕                  ▏ 188 MB/ 19 GB                  pulling manifest 
pulling eabc98a9bcbf:   1% ▕                  ▏ 235 MB/ 19 GB                  pulling manifest 
pulling eabc98a9bcbf:   2% ▕                  ▏ 334 MB/ 19 GB                  pulling manifest 
pulling eabc98a9bcbf:   2% ▕                  ▏ 422 MB/ 19 GB                  pulling manifest 
pulling eabc98a9bcbf:   2% ▕                  ▏ 468 MB/ 19 GB                  pulling manifest 
pulling eabc98a9bcbf:   3% ▕                  ▏ 562 MB/ 19 GB                  pulling manifest 
pulling eabc98a9bcbf:   3% ▕     

In [ ]:
subprocess.run(
    ['git', 'clone', 'https://github.com/Oyin-Adegoju/Formative_feedback_prototype.git',
     '/kaggle/working/repo'],
    capture_output=True, text=True
)
os.chdir('/kaggle/working/repo')
print("cwd:", os.getcwd())


In [ ]:
subprocess.run(['git', 'checkout', 'develop'], capture_output=True, text=True)
result = subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True)
print(result.stdout)


In [ ]:
import subprocess

# basismodel ophalen (eenmalig, ~19 GB — kan even duren)
subprocess.run(['ollama', 'pull', 'qwen2.5:32b'], capture_output=True, text=True)

# context window vergroten naar 32768
with open('/kaggle/working/Modelfile', 'w') as f:
    f.write("FROM qwen2.5:32b\nPARAMETER num_ctx 32768\n")

r = subprocess.run(['ollama', 'create', 'qwen2.5-32b-ctx', '-f', '/kaggle/working/Modelfile'],
                   capture_output=True, text=True)
print(r.stderr[-200:])   # je wilt 'success' zien
print(subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout)

In [7]:
subprocess.run(['git', 'pull', 'origin', 'develop'], capture_output=True, text=True, cwd='/kaggle/working/repo')
import os, subprocess
os.environ['LLM_BASE_URL'] = 'http://localhost:11434/v1'
os.environ['LLM_MODEL'] = 'qwen2.5-32b-ctx'
REPO = '/kaggle/working/repo'

# 1) Bouw de full-content handoff (CAPS-extractie, geen LLM)
subprocess.run(
    ['python', 'scripts/run_caps_on_anonymized.py',
     '--input', '/kaggle/input/datasets/oyinadegoju/baddie/96f07675_anonymized.json'],
    cwd=REPO, env=os.environ, check=True)

# 2) Draai de pipeline op die handoff
process = subprocess.Popen(
    ['python', 'scripts/run_full_pipeline_v2.py',
     '--handoff', 'data/anonymized/caps_handoff_96f07675.json',
     '--llm-timeout', '1800'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    cwd=REPO, env=os.environ,
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print(f"\nExit code: {process.returncode}")


  Full Pipeline v2: CAPS → Qwen quality → merge → feedback
  input        : data/anonymized/caps_handoff_96f07675.json  (handoff)
  LLM base URL : http://localhost:11434/v1
  LLM model    : qwen2.5-32b-ctx
  mode         : full pipeline


────────────────────────────────────────────────────────────────
  Step 1 — handoff aangeleverd (CAPS overgeslagen)
────────────────────────────────────────────────────────────────
  doc_id                  : 96f07675
  (CAPS overgeslagen — handoff rechtstreeks aangeleverd; structuur niet herbepaald.)

────────────────────────────────────────────────────────────────
  Step 2 — Qwen quality diagnosis
────────────────────────────────────────────────────────────────
  manual_review criteria: taalkeuze, security

────────────────────────────────────────────────────────────────
  Step 3 — Merge CAPS + Qwen
────────────────────────────────────────────────────────────────
  final_stoplight                 : red
  criteria_requiring_extra_review : taalkeuze, 